# This notebook is part of the LinkedIn Learning Course: [Generative AI: Introduction to Diffusion Models for Text Generation](https://www.linkedin.com/learning/generative-ai-introduction-to-diffusion-models-for-text-generation/building-a-basic-text-diffusion-model-29722011?resume=false&u=35754684)

It is part of the `Conclusion:Challenge - Use a predtrained diffusion model for text generation`.


## The Challenge:


Complete the following steps to predict the masked word in this sentence:

_“The future of AI is [MASK]”_


1. you start by importing the necessary libraries

2. Load the pre-trained model and tokenize.

3. Define the input, you create the string
  * You can name it input text or whatever name you prefer, and assign it to this value.

  * The future of AI is the mask that we are trying to predict.

4. You go ahead to tokenize the input, and then you run the model.

5. When you're running the model, the goal is to pass the tokenized input through the model to get logits.

  * Logits are predictions for each token, and they are the raw or normalized predictions scores for each token in the vocabulary.

6. Go higher to identify the mask token index, and then get the predicted token ID.
  * This is usually from the logits, you select the logits corresponding to the mask token, and then find the index of the token with the highest logit value.
  * This is usually the most preferable predicted token.

7. And then go ahead to decode the predicted token.

8. Use the tokenizer to convert the predicted token ID back into human-readable word

9. Print the predicted token.

10. That should give you the result.




In [28]:
#Checking the Python version - for this colab notebook, I use 3.12.11
!python --version

Python 3.12.11


# Loading Modules

* `AutoTokenizer`
  * AutoTokenizer is a component within the Hugging Face transformers library that loads the appropriate tokenizer for a given pre-trained model. It streamlines the crucial first step of a Natural Language Processing (NLP) pipeline, which is to convert raw text into a numerical format that a machine learning model can process.

* `ModernBertForMaskedLM` is a forward method class
  * The Large Language Model using an update version of BERT.
  * It incorporates several architectural improvements, such as:

    * Rotary Positional Embeddings, allowing for much longer sequence lengths.
    * More efficient handling of padding tokens.
    * Different activation functions (GeGLU layers).
    * Alternating attention mechanisms.


In [29]:
from transformers import AutoTokenizer, ModernBertForMaskedLM
import torch


# Load the Tokenizer and the Model

I use the `tommyp111/modernbert-diffusion` as suggested by the LinkedIn Instructor and Hugging Face's documentation:

[tommyp111/modernbert-diffusion](https://huggingface.co/tommyp111/modernbert-diffusion)

In [30]:
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("tommyp111/modernbert-diffusion") #answerdotai/ModernBERT-base

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [31]:
# create the model we want to use
model = ModernBertForMaskedLM.from_pretrained("tommyp111/modernbert-diffusion")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/792M [00:00<?, ?B/s]

---
# Prepare the Text Input

In [32]:
input_text = "The future of AI is [MASK]."

---
# Tokenize the Input

use the `return_tensor` = pt (pytorch)

In [33]:
inputs = tokenizer(input_text, return_tensors="pt")

---

## Make Predictiions

Pass the tokenized input to the model to get predictions

In [35]:
with torch.no_grad():
  outputs = model(**inputs) #unpacks list of inputs
  predictions = outputs.logits # Logits here are the raw/normalized prediction scores for each token

---
# Decode the Predictions

Identify the predicted token for the **masked** position and decode it back to a word.


## masked_index = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

identifying the column indices of masked tokens within a tensor of token IDs.

`inputs["input_ids"]`: This refers to a tensor containing the numerical IDs of tokens in a sequence, typically generated by a Hugging Face tokenizer.

`tokenizer.mask_token_id`: This provides the specific numerical ID that represents the mask token (e.g., [MASK]) in the vocabulary of the tokenizer.

`(inputs["input_ids"] == tokenizer.mask_token_id)`: This performs a boolean comparison, resulting in a boolean tensor of the same shape as inputs
["input_ids"]. True indicates the presence of a mask token, and False indicates the presence of any other token.

`.nonzero(as_tuple=True)`: This method returns the indices of the True values in the boolean tensor. When as_tuple=True, it returns a tuple of tensors, where each tensor corresponds to a dimension and contains the indices along that dimension. For a 2D tensor, it would return (row_indices_tensor, col_indices_tensor).

`[1]`: This accesses the second element of the tuple returned by nonzero(), which corresponds to the column indices of the True values (i.e., the masked tokens).

In essence, masked_index will contain a tensor of the column indices where the mask token ([MASK]) appears in the input_ids tensor. This is commonly used in tasks like Masked Language Modeling (MLM), where you need to know the positions of the masked tokens to predict their original values.

In [36]:
masked_index = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
masked_index



tensor([6])

## predicted_token_id = predictions[0, masked_index].argmax(axis=-1)

Determining the most likely predicted token for a specific masked position within a sequence, commonly found in natural language processing tasks like masked language modeling.


### Here's a breakdown:

* `predictions`: This likely represents the raw output (logits or probabilities) from a model, such as a transformer-based language model (e.g., BERT, GPT). This output typically has dimensions corresponding to batch size, sequence length, and vocabulary size.


* `predictions[0, masked_index]`: This selects a specific part of the predictions tensor.

* `0`: This indicates the first item in the batch. If multiple sequences were processed simultaneously, this selects the predictions for the first sequence.
masked_index: This refers to the specific position within that sequence where a token was masked (hidden) and the model is trying to predict it.

* `.argmax(axis=-1)`: This is the core operation for determining the predicted token.

  * `argmax()`: This function returns the index of the maximum value along a specified axis.

  * `axis=-1`: This specifies that the argmax operation should be performed along the last axis. In the context of predictions[0, masked_index], which now represents the predicted probabilities/logits for each vocabulary token at that specific masked position, axis=-1 means finding the index of the token with the highest probability/logit within the entire vocabulary.

`predicted_token_id`: The result of this operation is the predicted_token_id, which is the numerical ID of the token that the model predicts as the most likely replacement for the masked token at masked_index. This ID can then be mapped back to a human-readable word using a tokenizer's vocabulary.

In [37]:

predicted_token_id = predictions[0, masked_index].argmax(axis=-1)


## predicted_word = tokenizer.decode(predicted_token_id)

The code snippet `predicted_word = tokenizer.decode(predicted_token_id)` is performing a decoding operation in the context of Natural Language Processing (NLP), specifically with a tokenizer object from a library like Hugging Face Transformers.


### Here's a breakdown of what it does:

`predicted_token_id`: This represents a numerical ID (or a sequence of IDs) that a language model or other NLP system has predicted. In NLP, text is first converted into numerical representations (token IDs) for processing by models.

`tokenizer.decode()`: This method is part of a tokenizer's functionality. Its purpose is to reverse the tokenization process, converting the numerical predicted_token_id back into a human-readable string. This involves:

  Mapping the token ID(s) back to their corresponding tokens (e.g., subwords, words, or characters) based on the tokenizer's vocabulary.

  Reconstructing these tokens into a coherent string, often handling special tokens (like [CLS], [SEP]) and merging subword tokens back into full words, while also managing spacing and punctuation for readability.

`predicted_word`: This variable will store the resulting decoded string, representing the word or sequence of words that corresponds to the predicted_token_id.


In essence, this line of code takes a numerical output from an NLP model and transforms it back into a meaningful textual output, making the model's prediction interpretable. This is a crucial step in tasks like text generation, machine translation, or text summarization, where the model outputs numerical tokens that need to be converted back into natural language.

In [38]:
predicted_word = tokenizer.decode(predicted_token_id)
predicted_word

' here'

## Display the Result

Print out the predicted word to see the model's output



In [39]:
print(f"The predicted word is: {predicted_word}")


The predicted word is:  here


---
# Appplying Pipeline to see More results
This single result is not exactly good so, I am going to apply a `pipeline` approach as suggested by the LinkedIn Course to generate more possible results

In [40]:
from transformers import pipeline
unmasker = pipeline('fill-mask', model="tommyp111/modernbert-diffusion")
unmasker("The future of AI is [MASK]")

Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2509: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2509: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2509: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2509: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2509: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2509: UserWarning: Tesla T4 does not support 

[{'score': 0.25390625,
  'token': 1051,
  'token_str': '...',
  'sequence': 'The future of AI is...'},
 {'score': 0.08740234375,
  'token': 1060,
  'token_str': ' here',
  'sequence': 'The future of AI is here'},
 {'score': 0.046875,
  'token': 3346,
  'token_str': '...',
  'sequence': 'The future of AI is...'},
 {'score': 0.0439453125,
  'token': 8767,
  'token_str': ' uncertain',
  'sequence': 'The future of AI is uncertain'},
 {'score': 0.0341796875,
  'token': 6627,
  'token_str': ' bright',
  'sequence': 'The future of AI is bright'}]